# Smart Waste Segregation - classifier training on Colab

Trains and evaluates the 6-class TrashNet classifiers on a Colab GPU, then saves the
results back to Google Drive.

**Before you run this**, make sure of two things (see `docs/COLAB.md`):

1. `src/train_classifier.py` and `src/evaluate.py` are **committed and pushed** to GitHub -
   this notebook gets them by cloning the repo.
2. The TrashNet images are on your Drive (a zip is much faster than a folder).

**Runtime -> Change runtime type -> GPU** before starting.

Order of cells: config -> mount Drive -> clone + install -> copy data -> run configs -> zip to Drive.

## 0. Configuration

Everything you might need to change lives in this one cell.

In [ ]:
# ---- your repo ---------------------------------------------------------------
REPO_URL  = "https://github.com/aarya5p/smart-waste-management.git"
BRANCH    = "main"
REPO_IS_PRIVATE = False      # True -> you will be prompted for a GitHub token

# ---- where your data lives on Drive -------------------------------------------
# Either a .zip of the class folders (FAST, recommended) or the folder itself (slow).
# The zip/folder must contain the 6 class folders: cardboard, glass, metal, paper, plastic, trash
DRIVE_TRASHNET   = "/content/drive/MyDrive/minor_project/dataset-resized.zip"

# Where the result zips get written back to
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/minor_project/outputs"

# Resumable sweep store: each finished run is copied to <DRIVE_SWEEP_DIR>/<run_name>/ the moment
# it completes. On restart, any run whose folder is already here (with a DONE marker) is skipped.
DRIVE_SWEEP_DIR  = "/content/drive/MyDrive/minor_project/sweep"

# ---- local (fast) paths on the Colab VM ---------------------------------------
WORK_DIR  = "/content"
DATA_ROOT = "/content/data/raw/trashnet"     # the 6 class folders end up here

# ---- training defaults (per-config values below can override these) -----------
EPOCHS      = 30
BATCH_SIZE  = 32
NUM_WORKERS = 2        # Colab VMs usually have 2 vCPUs

# Every run is scored on BOTH splits, into results/<run_name>/val/ and results/<run_name>/test/.
# val  = the ONLY split you are allowed to make decisions from
# test = reported in the final tables, never used to change anything
EVAL_SPLITS = {"val": "splits/val.csv", "test": "splits/test.csv"}

# ---- dependency install strategy ----------------------------------------------
# Colab already ships torch/torchvision built for its exact CUDA driver, plus numpy,
# pandas, matplotlib, scipy and scikit-learn. Reinstalling the pinned versions from
# requirements.txt is slow and can break the CUDA build, so by default we install
# ONLY the packages Colab is missing. Set STRICT_REQUIREMENTS = True to force the
# exact pins from requirements.txt instead (slow, may require a runtime restart).
STRICT_REQUIREMENTS = False
EXTRA_PACKAGES = ["timm", "ultralytics", "imagehash", "grad-cam", "fvcore"]

import os
REPO_NAME = REPO_URL.rstrip("/").split("/")[-1].replace(".git", "")
REPO_DIR  = os.path.join(WORK_DIR, REPO_NAME)
print("repo dir :", REPO_DIR)
print("data root:", DATA_ROOT)

In [ ]:
import subprocess, sys, time
from pathlib import Path

def run(cmd, cwd=None, check=True):
    """Run a command and stream its output live (so you can watch training progress)."""
    print("$", " ".join(str(c) for c in cmd), flush=True)
    t0 = time.time()
    p = subprocess.Popen([str(c) for c in cmd], cwd=cwd, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    p.wait()
    dt = time.time() - t0
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {p.returncode} after {dt:.0f}s")
    print(f"[done in {dt:.0f}s]\n", flush=True)
    return p.returncode

## 1. Mount Google Drive and check the GPU

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU. Runtime -> Change runtime type -> T4 GPU, then re-run.")

## 2. Clone the repo and install dependencies

Re-running this cell pulls the latest commit instead of cloning again, so you can push a
fix from your laptop and just re-run it.

In [ ]:
import shutil

clone_url = REPO_URL
if REPO_IS_PRIVATE:
    from getpass import getpass
    token = getpass("GitHub personal access token (not stored, not printed): ")
    clone_url = REPO_URL.replace("https://", f"https://{token}@")

if Path(REPO_DIR, ".git").exists():
    print("Repo already present - pulling latest.")
    run(["git", "fetch", "origin", BRANCH], cwd=REPO_DIR)
    run(["git", "checkout", BRANCH], cwd=REPO_DIR)
    run(["git", "reset", "--hard", f"origin/{BRANCH}"], cwd=REPO_DIR)
else:
    run(["git", "clone", "--branch", BRANCH, clone_url, REPO_DIR], cwd=WORK_DIR)

del clone_url    # don't leave a tokenised URL lying around in the kernel

run(["git", "log", "-1", "--oneline"], cwd=REPO_DIR)

# Fail early and clearly if the training scripts were never pushed.
required = ["src/train_classifier.py", "src/evaluate.py",
            "splits/train.csv", "splits/val.csv", "splits/test.csv"]
missing = [f for f in required if not Path(REPO_DIR, f).exists()]
if missing:
    raise SystemExit(
        "These files are missing from the clone:\n  " + "\n  ".join(missing) +
        "\n\nCommit and push them from your laptop first:\n"
        "    git add src/ splits/ && git commit -m 'add training scripts' && git push"
    )
print("\nAll required files present.")

In [ ]:
if STRICT_REQUIREMENTS:
    # Exact pins from requirements.txt. Slow, and may force a runtime restart.
    run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], cwd=REPO_DIR)
else:
    # Only what Colab is missing; leave its CUDA-matched torch stack alone.
    run([sys.executable, "-m", "pip", "install", "-q", *EXTRA_PACKAGES], cwd=REPO_DIR)

# Record exactly what is installed, so the numbers in your report are traceable
# to a specific environment (Colab's versions differ from your laptop's).
freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout
Path(REPO_DIR, "pip_freeze_colab.txt").write_text(freeze)

import importlib
for mod in ["torch", "torchvision", "timm", "sklearn", "fvcore"]:
    try:
        m = importlib.import_module(mod)
        print(f"{mod:12s} {getattr(m, '__version__', 'n/a')}")
    except ImportError as e:
        print(f"{mod:12s} NOT INSTALLED ({e})")

## 3. Copy TrashNet from Drive to local disk

Training directly off `/content/drive` is slow: every image read goes through a network
filesystem, which starves the GPU. Copying the ~43 MB dataset to the VM's local disk once
takes a few seconds and makes every epoch faster.

In [ ]:
import zipfile

CLASSES = ["cardboard", "glass", "metal", "paper", "plastic", "trash"]
src = Path(DRIVE_TRASHNET)
dst = Path(DATA_ROOT)

if not src.exists():
    raise SystemExit(f"Not found on Drive: {src}\nFix DRIVE_TRASHNET in the config cell.")

if dst.exists() and all((dst / c).is_dir() for c in CLASSES):
    print("Dataset already on local disk - skipping copy.")
else:
    dst.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    if src.suffix.lower() == ".zip":
        print(f"Unzipping {src} ...")
        with zipfile.ZipFile(src) as z:
            z.extractall("/content/_trashnet_tmp")
        # The zip may hold the class folders directly, or nest them one level down
        # (e.g. dataset-resized/cardboard/...). Find whichever level has the classes.
        root = None
        for cand in [Path("/content/_trashnet_tmp"), *Path("/content/_trashnet_tmp").iterdir()]:
            if cand.is_dir() and all((cand / c).is_dir() for c in CLASSES):
                root = cand
                break
        if root is None:
            raise SystemExit("Could not find the 6 class folders inside the zip.")
        for c in CLASSES:
            shutil.copytree(root / c, dst / c, dirs_exist_ok=True)
        shutil.rmtree("/content/_trashnet_tmp", ignore_errors=True)
    else:
        print(f"Copying folder {src} ... (slow over Drive)")
        for c in CLASSES:
            if not (src / c).is_dir():
                raise SystemExit(f"Missing class folder on Drive: {src/c}")
            shutil.copytree(src / c, dst / c, dirs_exist_ok=True)
    print(f"Copied in {time.time()-t0:.0f}s")

# Drop macOS junk that would otherwise be scanned as images
for junk in dst.rglob(".DS_Store"):
    junk.unlink()

total = 0
for c in CLASSES:
    n = len(list((dst / c).glob("*.jpg")))
    total += n
    print(f"  {c:10s} {n:4d}")
print(f"  {'TOTAL':10s} {total:4d}   (expected 2527)")

In [ ]:
# Sanity gate: every path in the committed splits must exist locally.
# Catches a wrong Drive path or a half-finished copy BEFORE you burn GPU time.
import pandas as pd

for split in ["train", "val", "test"]:
    df = pd.read_csv(Path(REPO_DIR, "splits", f"{split}.csv"))
    missing = [p for p in df["filepath"] if not (Path(DATA_ROOT) / p).exists()]
    print(f"{split:5s} {len(df):5d} rows | missing files: {len(missing)}")
    if missing:
        raise SystemExit(f"{len(missing)} files from {split}.csv are not on disk, "
                         f"e.g. {missing[:3]}")
print("\nAll split files present - safe to train.")

## 4. Define the runs (Phase 3 frozen sweep)

All 5 classifiers x seeds 42, 56, 99, mode `frozen` (CLAUDE.md rule 5). Order is **seed-major**:
every model with seed 42 first, then every model with seed 56, then seed 99 - so if the session
dies part-way, you still have every model covered. `mobilenet_v2` seed 42 is already done and
imported into the repo, so it is skipped (`ALREADY_DONE`).

`model`, `mode` and `seed` are required per entry; `epochs`, `batch_size`, `lr`, `img_size`,
`patience` are optional overrides.

In [ ]:
MODELS = ["mobilenet_v2", "mobilenet_v3_large", "efficientnet_b0", "resnet50", "convnext_tiny"]
SEEDS  = [42, 56, 99]

# Runs that were already trained/evaluated/imported elsewhere - do not redo them.
ALREADY_DONE = {("mobilenet_v2", "frozen", 42)}

# Seed outer loop, model inner loop -> all seed-42 runs, then 56, then 99.
CONFIGS = [
    {"model": m, "mode": "frozen", "seed": s}
    for s in SEEDS
    for m in MODELS
    if (m, "frozen", s) not in ALREADY_DONE
]

# --- fine-tuned subset (later; choose the 2-3 models from VAL results of the frozen sweep) ---
# CONFIGS += [
#     {"model": m, "mode": "finetune", "seed": s}
#     for s in SEEDS for m in ["<model chosen from val>", ...]
# ]

for c in CONFIGS:
    print(f"{c['model']}_{c['mode']}_seed{c['seed']}")
print(f"\n{len(CONFIGS)} run(s) queued (finished ones on Drive are skipped when you run the next cell)")

### Run them (resumable)

For each config, in order:

1. **Skip** if `DRIVE_SWEEP_DIR/<run_name>/DONE` exists (it finished in an earlier session).
2. Train (`splits/train.csv`, early stopping on val loss only), then evaluate on **val and test**.
3. **Immediately** copy `runs/<run_name>/`, `results/<run_name>/` and the LOG.md rows this run
   appended to `DRIVE_SWEEP_DIR/<run_name>/`. It is copied to a `.partial` folder and renamed
   at the end, with `DONE` written last, so a disconnect mid-copy never leaves something that
   looks finished.

A run that crashes is reported at the end and the sweep continues; just re-run this cell to retry it.
If Colab disconnects: re-run cells from the top (config -> mount -> clone -> data -> CONFIGS -> this
cell). Finished runs are skipped. Only the run that was in progress is lost.

> **Decision rule (CLAUDE.md rule 2).** Every decision is made from the **validation** columns
> only. Test columns are for the final report tables and must never change anything.

In [ ]:
import json, os, shutil
from datetime import datetime

SWEEP = Path(DRIVE_SWEEP_DIR)
SWEEP.mkdir(parents=True, exist_ok=True)
LOG_PATH = Path(REPO_DIR, "LOG.md")


def name_of(cfg):
    return f"{cfg['model']}_{cfg['mode']}_seed{cfg['seed']}"


def done_on_drive(rn):
    d = SWEEP / rn
    return (d / "DONE").exists() and all((d / "results" / rn / s / "metrics.json").exists()
                                         for s in EVAL_SPLITS)


def save_to_drive(rn, log_rows):
    """Copy one finished run to Drive. .partial folder + rename + DONE written last."""
    tmp, final = SWEEP / f"{rn}.partial", SWEEP / rn
    shutil.rmtree(tmp, ignore_errors=True)
    shutil.copytree(Path(REPO_DIR, "runs", rn), tmp / "runs" / rn)
    shutil.copytree(Path(REPO_DIR, "results", rn), tmp / "results" / rn)
    (tmp / "log_rows.md").write_text("".join(log_rows))
    (tmp / "DONE").write_text(datetime.now().isoformat())
    shutil.rmtree(final, ignore_errors=True)      # only a stale, not-DONE folder can be here
    tmp.rename(final)


results_summary, failures, skipped = [], [], []

for i, cfg in enumerate(CONFIGS, 1):
    rn = name_of(cfg)
    print("=" * 78)
    if done_on_drive(rn):
        print(f"[{i}/{len(CONFIGS)}] {rn}: already on Drive - SKIPPED", flush=True)
        skipped.append(rn)
        continue
    print(f"[{i}/{len(CONFIGS)}] {rn}")
    print("=" * 78, flush=True)

    try:
        n_log_before = len(LOG_PATH.read_text().splitlines()) if LOG_PATH.exists() else 0

        train_cmd = [sys.executable, "src/train_classifier.py",
                     "--data_root", DATA_ROOT,
                     "--model", cfg["model"], "--mode", cfg["mode"], "--seed", cfg["seed"],
                     "--epochs", cfg.get("epochs", EPOCHS),
                     "--batch_size", cfg.get("batch_size", BATCH_SIZE),
                     "--num_workers", NUM_WORKERS]
        for opt in ["lr", "img_size", "patience"]:
            if opt in cfg:
                train_cmd += [f"--{opt}", cfg[opt]]
        run(train_cmd, cwd=REPO_DIR)

        ckpt = Path(REPO_DIR, "runs", rn, "best.pt")
        if not ckpt.exists():
            raise RuntimeError(f"No checkpoint produced at {ckpt}")

        row = {"run": rn}
        for split, csv_path in EVAL_SPLITS.items():
            out_dir = f"results/{rn}/{split}"
            print(f"--- evaluating on {split} ({csv_path}) ---", flush=True)
            run([sys.executable, "src/evaluate.py",
                 "--checkpoint", f"runs/{rn}/best.pt", "--data_root", DATA_ROOT,
                 "--csv", csv_path, "--out_dir", out_dir, "--num_workers", NUM_WORKERS],
                cwd=REPO_DIR)
            m = json.loads(Path(REPO_DIR, out_dir, "metrics.json").read_text())
            row[f"{split}_acc"] = round(m["accuracy"], 4)
            row[f"{split}_macro_f1"] = round(m["macro_f1"], 4)

        new_rows = [l for l in LOG_PATH.read_text().splitlines(keepends=True)[n_log_before:]
                    if l.startswith("|")]
        save_to_drive(rn, new_rows)
        print(f"Saved {rn} to Drive ({len(new_rows)} LOG rows)\n", flush=True)
        results_summary.append(row)
    except Exception as e:        # one bad run must not kill the sweep; re-run the cell to retry
        print(f"!!! {rn} FAILED: {e}\n", flush=True)
        failures.append(rn)

print("\nSkipped (already on Drive):", skipped or "none")
print("Failed (re-run this cell to retry):", failures or "none")
if results_summary:
    print("\nRuns finished this session:")
    print(pd.DataFrame(results_summary).to_string(index=False))
    print("\nDecide from the val_* columns ONLY. The test_* columns are for the final "
          "report tables and must not change any choice you make.")

## 5. Bundle everything on Drive into one zip

`DRIVE_SWEEP_DIR` is the source of truth (it survives disconnects). This cell merges **every**
finished run there - from this session and earlier ones - into one
`waste_results_<timestamp>.zip` in `DRIVE_OUTPUT_DIR`, containing `runs/`, `results/`,
`LOG_rows.md` (all runs' LOG rows, concatenated) and `pip_freeze_colab.txt`.
Run it whenever you want a download; it is safe to run on a partial sweep.

In [ ]:
import zipfile
from datetime import datetime

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
zip_path = Path(WORK_DIR, f"waste_results_{stamp}.zip")

finished = sorted(d for d in SWEEP.iterdir() if d.is_dir() and (d / "DONE").exists())
print(f"{len(finished)} finished run(s) on Drive:")
for d in finished:
    print("  ", d.name)

n, log_rows = 0, []
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for d in finished:
        for p in d.rglob("*"):
            if p.is_file() and p.name not in ("DONE", "log_rows.md"):
                z.write(p, p.relative_to(d))
                n += 1
        log_rows += (d / "log_rows.md").read_text().splitlines(keepends=True)
    z.writestr("LOG_rows.md", "".join(log_rows))
    freeze = Path(REPO_DIR, "pip_freeze_colab.txt")
    if freeze.exists():
        z.write(freeze, "pip_freeze_colab.txt")

print(f"{n} files -> {zip_path} ({zip_path.stat().st_size/1024**2:.1f} MB)")
out_dir = Path(DRIVE_OUTPUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)
dest = out_dir / zip_path.name
shutil.copy2(zip_path, dest)
print("Saved to Drive:", dest)

## After the run

1. Download the zip from `DRIVE_OUTPUT_DIR` and unzip `runs/` and `results/` into the repo.
2. Append the rows in the zip's `LOG_rows.md` to the repo's `LOG.md` (the table format is
   identical, so it is a plain append; skip rows for runs already in the repo's LOG.md). The Colab
   VM's LOG.md disappears with the session (rule 6). There are two eval rows per run (val, test).
3. Run `python src/summarize_results.py` locally -> `results/summary_frozen.csv` and `.md`
   (mean ± std over seeds; this includes the imported mobilenet_v2 seed 42). Commit `results/`.
   `runs/` holds checkpoints and stays untracked.

**Latency caveat (rule 8):** any latency in `metrics.json` from this notebook was measured on
a Colab GPU and is stamped `counts_for_report: false`. Only Raspberry Pi latency goes in the report.